# Lab 2\. Applying torch.compile and Observing the Compilation-Speedup Pattern

## Overview

In the chapter you saw that PyTorch runs a model one operation at a time in eager mode, and that `torch.compile` can make the same model run faster by capturing its operations into a graph, optimizing that graph, and reusing the compiled result on later calls. You also saw the pattern that surprises people the first time: the compiled model is slow on its very first call, because compilation happens then, and faster on the calls that follow. This lab is where you watch that pattern happen on a model you run yourself.

You will define a small image classifier, run it in ordinary eager mode and time it, then apply `torch.compile` with a single line and run it again. You will time the first compiled call on its own so the compilation cost is visible, then time the calls after that to see the steady-state speed. Finally you will compare the eager time against the compiled steady-state time and interpret what you find for the hardware you are on.

Being able to measure this yourself matters in real work. Before you decide to compile a model in a training or serving pipeline, you need to know how to confirm that compilation actually helps for your model and your hardware, and how to read the first-run cost so you do not mistake it for a failure. This lab builds that habit on a small, safe example so the reasoning transfers to the larger models where the gains are bigger.

> **Note:** Absolute timing numbers depend heavily on your hardware. On a small model running on a CPU, the compiled steady-state time may be close to the eager time, or even slower, because a small model gives the compiler little to fuse while still charging the one-time compile cost. That outcome is itself a valid result: the transferable lesson is the *shape* of what you observe (a slow first call, then a steady state), not a specific speedup number on a laptop.

## Learning Objectives

By the end of this lab, you should be able to:

- Apply `torch.compile` to an existing PyTorch model using the one-line API, leaving the original model unchanged  
- Run a model in eager mode and in compiled mode and time both with a simple, repeatable benchmark  
- Observe the first-run compilation overhead and explain why the first compiled call is slower than the calls that follow  
- Measure the steady-state result on repeated calls and interpret it correctly for your model and hardware

## Prerequisites

This course’s labs are **not cumulative**, so this lab does not depend on any prior resources or the state of earlier labs if there are any.

You need the following before you start:  
**A notebook environment:** either a Google account for Google Colab, or a local installation of Jupyter Notebook. Every step in this lab runs on a CPU; no GPU is required.

- **Python 3.12 or later.** On Colab this is already provided. For a local environment, confirm your version with `python --version`.  
- **PyTorch 2.12.1,** installed with `pip` in the first setup step below. If you are on Colab, it may already be present; the setup step confirms the version.  
- **A C++ compiler available on the machine.** On CPU, `torch.compile`'s default backend generates and builds C++ code, so a working C++ compiler (for example, `g++`) and the Python development headers must be present. Google Colab includes these by default. If you prefer to run locally, follow C++ installation instructions for your operating system \[[Windows](https://docs.pytorch.org/tutorials/unstable/inductor_windows.html), [MacOS](https://www.cs.rhodes.edu/~kirlinp/courses/cs2/s17/installing-clion/xcode.html), [Ubuntu](https://ubuntu.com/developers/docs/howto/gcc-setup/)\].  
- **Knowledge:** You should be comfortable importing libraries, defining a simple model or function in Python, and running notebook cells. No prior experience with `torch.compile` is assumed.

## Estimated Time

20–30 minutes.

## How to Perform the Exercise

You will work entirely in one notebook, running cells from top to bottom. Create a new notebook in Google Colab (**File \> New notebook**) or a new `.ipynb` in local Jupyter. Download the chapter 2 lab notebook from the [LFD2003-resources GitHub repository](https://github.com/lftraining/LFD2003-resources.git) from the **LFD2003-resources/labs** directory. In Google Colab, select File, then Upload Notebook, and select the chapter 2 lab notebook you downloaded.

Read the explanation before running each cell, run it, and compare what you see to the Sample Output or Output shown after each block. Sample Output means the exact values will differ from run to run (random tensors, dataset counts, file paths), so match the shape and structure rather than every character. Output marks a checkpoint where a specific outcome confirms the step worked.

Commands in this lab use the notebook form, where a line beginning with `!` runs a shell command from inside a notebook cell (for example, `!pip install ...`). If you run these in a local terminal instead of a notebook cell, drop the leading `!`.

> **Note:** Run the cells in order and only once each, unless a step says otherwise. If you restart the notebook's runtime, re-run the cells from the top, because the variables live in memory and are cleared on restart.

## Setup: Environment Preparation

The steps in this section only prepare the environment. They are not the learning content of the lab, and on a managed course platform they may already be done for you. Run them once at the top of your notebook, then move on to Step 1\.

### Setup 1\. Install torch and torchvision

Install the library the lab needs. On Colab a recent version is usually already present, in which case pip reports that the requirement is already satisfied and does nothing further.


In [ ]:
!pip install torch


The leading `!` tells the notebook to run this as a shell command rather than as Python.

### Setup 2\. Import PyTorch and Confirm the Version

Begin by importing the libraries the lab uses and confirming that your PyTorch version supports `torch.compile`. You need `torch` for the model and compilation, `torch.nn` for the layers, and Python's built-in `time` module for the benchmark you will write later.


In [ ]:
import time
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


The version check matters because `torch.compile` is only available in PyTorch 2.0 and later; if the printed version starts with a number below 2, the rest of the lab will not run. The second line reports whether a GPU is available. For the main lab you do not need one — the value simply tells you which device you are working on.

**Sample Output**

```
PyTorch version: 2.11.0+cpu
CUDA available: False
```

### Step 1\. Select the Device

Choose the device the lab will run on. This lab is written for CPU so that everyone can complete it, and the same code runs unchanged on a GPU if one is present.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Selecting the device once and reusing the `device` variable keeps the model and its inputs on the same device, which is required for them to work together. On the free tier of Colab without a GPU runtime enabled, this prints `cpu`, which is expected.

**Sample Output**

```
Using device: cpu
```

## Building and Timing the Model

### Step 2\. Define a Small Image Classifier

Define the model you will compile. This is a small convolutional classifier shaped for 28-by-28 grayscale images, the size of handwritten-digit images, so it stands in for the kind of small digit classifier described in the chapter.


In [ ]:
class SmallClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SmallClassifier().to(device)
model.eval()


The model has two convolution-and-pooling stages followed by two linear layers, ending in ten outputs, one per digit class. It has more than one operation, which is what gives `torch.compile` something to work with when it captures and fuses the graph. Calling `model.eval()` puts the model in evaluation mode, which is the right setting for timing the forward pass, because you are measuring execution speed rather than training the model.

> **Note:** This lab uses random input tensors rather than the real handwritten-digit dataset. The chapter referred to a trained digit classifier; here you use random inputs of the same shape because the goal is to measure execution time, not classification accuracy. Random data of the correct shape exercises exactly the same operations, and it keeps the lab self-contained with no dataset to download.

### Step 3\. Create an Input Batch

Create a dummy batch of random inputs shaped like a set of 28-by-28 grayscale images. The four numbers in the shape are the batch size, the number of channels, the image height, and the image width.


In [ ]:
x = torch.randn(64, 1, 28, 28, device=device)
print("Input shape:", tuple(x.shape))


A batch size of 64 gives the model enough work per call that timing is meaningful, without being so large that a CPU run becomes slow. Placing the input on the same `device` as the model, using `device=device`, is what lets the two work together.

**Sample Output**

```
Input shape: (64, 1, 28, 28)
```

### Step 4\. Write a Simple Benchmark Function

Write a small function that times how long the model takes per call, averaged over several calls. Timing a single call is unreliable, so the function first runs the model a few times to warm up, then times a batch of calls and returns the average.


In [ ]:
def benchmark(run_once, device='cpu', warmup=3, iters=10):
    # Warm-up calls are not timed; they let one-time setup settle.
    device = torch.device(device)
    uses_cuda = device.type == "cuda"

    for _ in range(warmup):
        run_once()
    if uses_cuda:
        torch.cuda.synchronize(device)

    start = time.perf_counter()
    for _ in range(iters):
        run_once()
    if uses_cuda:
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - start

    return elapsed / iters


Three parts of this function are worth understanding. The warm-up loop runs the model a few times without timing it, so that any one-time setup does not land inside the measured window. The `torch.cuda.synchronize()` calls only do something on a GPU: because the GPU runs work asynchronously, you have to wait for it to finish before you read the clock, or you would time only the launch and not the work. On CPU these lines are skipped, because CPU execution is not asynchronous in this way. The function returns the average seconds per call, which is the number you will compare between eager and compiled runs.

> **Note:** PyTorch also ships an official benchmarking helper, `torch.utils.benchmark.Timer`, which handles warm-up and synchronization for you. This lab uses a hand-written function instead so that each part of the measurement is visible. The manual version is enough for this comparison.

### Step 5\. Measure the Eager Baseline

Measure how long the model takes per call in ordinary eager mode, before any compilation. This is the baseline you will compare against. Wrap the call in `torch.no_grad()` so the timing reflects the forward pass without building gradients, which is the right comparison for execution speed.


In [ ]:
def run_eager():
    with torch.no_grad():
        model(x)


eager_times = []
for i in range(30):
  eager_times.append(benchmark(run_eager, device))

eager_time = sum(eager_times) / len(eager_times)
print(f"Eager mode: {eager_time * 1000:.2f} ms per call")


The number printed here is your reference point. Keep it in mind for the comparison in Step 9\.

**Sample Output**

```
Eager mode: 28.77 ms per call
```

## Compiling the Model

### Step 6\. Apply torch.compile

Apply compilation to the model with the one-line API. `torch.compile` takes your model and returns a compiled version. Your original `model` is not changed, so you can still use it; the compiled version is a separate object you call the same way.


In [ ]:
compiled_model = torch.compile(model)


Nothing is compiled yet at this point. `torch.compile` returns almost immediately, because the actual work — capturing the graph, optimizing it, and generating code — is deferred until the first time you run `compiled_model` on data. You will see that deferred cost in the next step.

### Step 7\. Run the First Compiled Call and Watch the Overhead

Run the compiled model once and time that single call on its own. This first call is where graph capture, optimization, and code generation all happen, so it carries the full compilation cost.


In [ ]:
def run_compiled():
    with torch.no_grad():
        compiled_model(x)

start = time.perf_counter()
run_compiled()          # first call — includes compilation
if device == "cuda":
    torch.cuda.synchronize(device)
first_call_time = time.perf_counter() - start
print(f"First compiled call (includes compilation): {first_call_time * 1000:.2f} ms")


Expect this number to be much larger than the eager per-call time from Step 6, often by a wide margin. That is not an error and not a sign that compilation made the model slower. It is the one-time cost of compiling, paid on this first call and not on the calls that follow.

**Sample Output**

```
First compiled call (includes compilation): 4211.36 ms
```

> **Note:** The first compiled call can take several seconds, especially on CPU where the backend builds C++ code. If the cell seems to pause, it is compiling, not frozen. Let it finish.

### Step 8\. Measure the Compiled Steady State and Compare

Now that compilation has happened, measure the compiled model the same way you measured eager mode. Because the compiled code already exists and gets reused, these calls should no longer pay the compilation cost.


In [ ]:

compiled_times = []
for i in range(30):
  compiled_times.append(benchmark(run_compiled, device))

compiled_time = sum(compiled_times) / len(compiled_times)
print(f"Eager mode:            {eager_time * 1000:.2f} ms per call")
print(f"Compiled steady state: {compiled_time * 1000:.2f} ms per call")
print(f"Speedup: {eager_time / compiled_time:.2f}x")


Read the three numbers together. The compiled steady-state time is the fair comparison against eager, because both now exclude one-time setup. A speedup above `1.00x` means the compiled model is faster per call; a value near or below `1.00x` means compilation did not help for this model on this hardware.

**Sample Output**

```
Eager mode:            28.77 ms per call
Compiled steady state: 15.41 ms per call
Speedup: 1.87x
```

Interpret your result honestly against what the chapter set up. On a small model, a small speedup, no change, or even a slight slowdown are all realistic outcomes, because a small model gives the compiler little to fuse while still paying the compile cost once. The result that always holds, and the one this lab is really about, is the contrast between Step 8 and Step 9: the first call is expensive, and the calls after it are not. That is the compile-once-then-reuse pattern, and it is what makes compilation pay off on larger models that run many times.

## Optional: Try It on a GPU

This section is optional and is included only so you can contrast CPU behavior with GPU behavior. It is not required to meet the lab's objectives, and you can skip it if you do not have a GPU.

If you are in Google Colab, you can switch to a free GPU runtime through the menu: select **Runtime**, then **Change runtime type**, then choose a GPU hardware accelerator, and save. Changing the runtime restarts it, so you will need to run the notebook again from the top.

After the runtime restarts with a GPU, run all the cells from Step 1 again. Step 2 will now print `Using device: cuda`, and the timing steps will run on the GPU. Compare the speedup you see here against the one you saw on CPU. Compilation can be especially helpful on a GPU, because it fuses operations, reduces kernel-launch overhead, and avoids writing intermediate results to GPU memory. However, for a small model, compilation may produce a larger relative speedup on CPU because Python and operator-dispatch overhead can account for a greater fraction of the total execution time. The sample output below illustrates this well. In both runs, eager and compiled, GPU remains faster in absolute terms but the speedup is smaller than on CPU.

**Sample Output**

```
Eager mode:            0.46 ms per call
Compiled steady state: 0.40 ms per call
Speedup: 1.14x
```

## Cleanup

This lab creates no files and no lasting resources. Everything you made — the model, the input batch, and the compiled model — lives in the notebook's memory only.

To release that memory and clear the compilation cache, either restart the runtime (Colab's "Restart runtime" or Jupyter's "Restart kernel"), or reset the compiler state and drop the references in a cell:


In [ ]:
torch._dynamo.reset()   # clears torch.compile's cached compiled code
del compiled_model, model


> **Note:** `torch._dynamo.reset()` clears the compiled code that `torch.compile` cached during this lab. Run it only when you want to start the compilation experiment fresh; after resetting, the next compiled call will pay the first-run compilation cost again.

On Google Colab the runtime is temporary. When your Colab session ends, everything is cleared.

## Summary

In this lab you applied `torch.compile` to a small classifier with a single line, timed the model before and after compilation, and separated the expensive first compiled call from the cheaper calls that followed. That separation is the heart of what you learned by doing: compilation is a one-time cost paid on the first run and repaid on every run after, which is why the fair comparison uses the compiled steady-state time rather than the first call. You also saw, on a small model and likely on CPU, that the steady-state speedup can be modest or absent — a concrete reminder that compilation is a tool to reach for when a model is large enough and runs often enough to repay the compile cost, not a switch that always makes things faster. The habit you practiced, measuring both modes and reading the numbers before deciding, is exactly what you will use before turning on compilation in a real training or serving pipeline.  
